# OMNIDRIVE - Stage 3: Reasoning Module Fine-Tuning on Colab T4

This notebook fine-tunes a Vision-Language-Action (VLA) model (like LLaVA or Alpamayo) to handle **rare, long-tail scenarios** (e.g., soldier hand signals, flooded roads, unique checkpoints).

To make a massive 7B-8B parameter model fit onto the free 16GB Colab T4 GPU, we use **4-bit Quantization** and **LoRA** (Low-Rank Adaptation), which only trains ~0.1% of the network.

In [ ]:
# 1. MOUNT GOOGLE DRIVE & INSTALL DEPENDENCIES
# ============================================
import os

from google.colab import drive

print("Mounting Google Drive...")
drive.mount('/content/drive')

# Create directories for VLA checkpoints and dataset
VLA_CHECKPOINT_DIR = '/content/drive/MyDrive/OMNIDRIVE_PROJECT/checkpoints/reasoning'
VLA_DATA_DIR = '/content/drive/MyDrive/OMNIDRIVE_PROJECT/data/rare_scenarios'
os.makedirs(VLA_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(VLA_DATA_DIR, exist_ok=True)

print("\nInstalling advanced training libraries (PEFT, BitsAndBytes)...")
!pip install -q torch torchvision transformers accelerate bitsandbytes peft datasets
print("✅ Dependencies installed!")

In [ ]:
# 2. SETUP OMNIDRIVE REPOSITORY
# =============================
import shutil
import sys

OMNIDRIVE_SRC = '/content/OMNIDRIVE_PROJECT'

if not os.path.exists(OMNIDRIVE_SRC):
    print("Copying OMNIDRIVE_PROJECT from your Google Drive...")
    drive_project_path = '/content/drive/MyDrive/OMNIDRIVE_PROJECT'
    if os.path.exists(drive_project_path):
        shutil.copytree(drive_project_path, OMNIDRIVE_SRC, dirs_exist_ok=True)
        print("✅ Copied source code.")

src_path = os.path.join(OMNIDRIVE_SRC, 'src')
if src_path not in sys.path:
    sys.path.append(src_path)
print("✅ Python path configured.")

In [ ]:
# 3. LOAD MODEL IN 4-BIT PRECISION
# ================================
# We load LLaVA-1.6 (or Alpamayo) using BitsAndBytes.
# This squashes the 16GB model down to ~5GB of VRAM.

import torch
from transformers import AutoProcessor, BitsAndBytesConfig, LlavaNextForConditionalGeneration

print("Configuring 4-bit Quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# We use LLaVA as the open-source VLA foundation
model_id = "llava-hf/llava-v1.6-mistral-7b-hf"

print(f"\n📥 Downloading & Loading {model_id}...")
print("This takes ~5 minutes as it downloads 14GB of weights.")

processor = AutoProcessor.from_pretrained(model_id)
model = LlavaNextForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print("\n✅ Model loaded successfully into 4-bit space!")

In [ ]:
# 4. APPLY LoRA (Low-Rank Adaptation)
# ===================================
# We freeze the 7 Billion parameters and only train tiny adapter layers.

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print("Preparing model for k-bit training...")
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], # Target attention blocks
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

print("\n✅ LoRA Applied!")
model.print_trainable_parameters()
# You should see that less than 0.2% of parameters are trainable!

In [ ]:
# 5. PREPARE THE DATASET
# ======================
# The Reasoning Module needs image+text pairs to learn from.

import json

import torch.utils.data as data
from PIL import Image

# Create a dummy dataset file if you don't have one yet
dummy_data_path = f"{VLA_DATA_DIR}/rare_scenarios.json"
if not os.path.exists(dummy_data_path):
    dummy_data = [
        {
            "image": "soldier_stop.jpg",
            "prompt": "[INST] <image>\nWhat should the vehicle do? [/INST]",
            "completion": "STOP. A uniformed soldier is displaying a hand stop signal directly in the vehicle's path."
        }
    ]
    with open(dummy_data_path, 'w') as f:
        json.dump(dummy_data, f)
    print(f"⚠️ Created a template dataset at {dummy_data_path}")
    print("You need to replace this with your actual images and JSON labels!")

class VLADataset(data.Dataset):
    def __init__(self, json_path, img_dir, processor):
        with open(json_path) as f:
            self.samples = json.load(f)
        self.img_dir = img_dir
        self.processor = processor

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        try:
            img_path = os.path.join(self.img_dir, sample['image'])
            image = Image.open(img_path).convert('RGB')
        except:
            # Create dummy image if real one isn't found for notebook testing
            image = Image.new('RGB', (224, 224), color = (73, 109, 137))

        text = sample['prompt'] + " " + sample['completion']

        inputs = self.processor(text=text, images=image, return_tensors="pt", padding="max_length", max_length=128, truncation=True)

        # Remove batch dimension added by processor
        item = {k: v.squeeze(0) for k, v in inputs.items()}
        # The labels for language modeling are the input_ids themselves
        item["labels"] = item["input_ids"].clone()
        # Ignore padding tokens in loss calculation
        item["labels"][item["attention_mask"] == 0] = -100

        return item

dataset = VLADataset(dummy_data_path, VLA_DATA_DIR, processor)
print(f"✅ Dataset loaded with {len(dataset)} examples.")

In [ ]:
# 6. TRAINING LOOP
# ================
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir=VLA_CHECKPOINT_DIR,
    per_device_train_batch_size=2,  # Tiny batch size for T4
    gradient_accumulation_steps=8,  # Simulate larger batch size
    learning_rate=2e-4,
    fp16=True,                      # Mixed precision
    logging_steps=10,
    max_steps=200,                  # Short training run for demonstration
    save_strategy="steps",
    save_steps=100,
    optim="paged_adamw_8bit",       # Memory efficient optimizer
)

# Data collator to batch our items
def collate_fn(examples):
    import torch
    batch = {}
    for k in examples[0].keys():
        batch[k] = torch.stack([ex[k] for ex in examples])
    return batch

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=collate_fn,
)

print("\n🚀 Starting LoRA Fine-Tuning...")
try:
    # Uncomment to actually run training when images are uploaded!
    # trainer.train()
    print("Training logic is ready! Upload your images to VLA_DATA_DIR and uncomment trainer.train()")
except Exception as e:
    print(f"Training error (expected if dummy data): {e}")

In [ ]:
# 7. SAVE FINAL ADAPTERS
# ======================
# When training is done, we save the LoRA adapters to Google Drive.

final_save_path = f"{VLA_CHECKPOINT_DIR}/omnidrive_reasoning_v1"
# trainer.model.save_pretrained(final_save_path)
# processor.save_pretrained(final_save_path)

print(f"\n💾 Final adapters will be saved to: {final_save_path}")
print("These adapter weights are tiny (~20MB) and sit on top of the base LLaVA model during inference in the car.")